# Cyclistic Bikeshare Case Study: Data Cleaning & Preparation

**Business task:** How do annual members and casual riders use Cyclistic bikes differently?

**Data source:** 12 months (Jan–Dec 2025) of public Divvy trip data, downloaded from the
[Divvy trip data repository](https://divvy-tripdata.s3.amazonaws.com/index.html).

**Tools:** Python (pandas) for cleaning and feature engineering. Cleaned output is later
analyzed in Tableau and cross-validated in BigQuery SQL.

This notebook covers the **Prepare** and **Process** phases of the analysis: loading,
combining, cleaning, and transforming 12 monthly CSV files (~5.5M rows) into a single
analysis-ready dataset.

## Load and Combine Data
Mount Google Drive, locate the 12 monthly CSV files, and combine them into a single
DataFrame. Column headers are checked for consistency across all 12 files before
combining (not shown as a separate step here, but verified manually in MS Excel) to avoid silent
schema mismatches.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import glob
csv_files = glob.glob("/content/drive/MyDrive/raw data/*.csv")
print(csv_files)
print(len(csv_files))

['/content/drive/MyDrive/raw data/202501-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202502-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202503-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202504-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202505-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202506-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202507-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202508-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202509-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202510-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202511-divvy-tripdata.csv', '/content/drive/MyDrive/raw data/202512-divvy-tripdata.csv']
12


In [ ]:
import pandas as pd
df_list = []
for f in csv_files:
    print("reading", f)
    df = pd.read_csv(f, parse_dates=["started_at", "ended_at"])
    df_list.append(df)

all_trips = pd.concat(df_list, ignore_index=True)
print(all_trips.shape)

reading /content/drive/MyDrive/raw data/202501-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202502-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202503-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202504-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202505-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202506-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202507-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202508-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202509-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202510-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202511-divvy-tripdata.csv
reading /content/drive/MyDrive/raw data/202512-divvy-tripdata.csv
(5552994, 13)


## Initial Cleaning

Remove exact duplicate rows, duplicate `ride_id` values (each ride should be unique),
and rows missing the fields required for this analysis (`started_at`, `ended_at`,
`member_casual`).

In [ ]:
# Drop exact duplicate rows
before = len(all_trips)
all_trips = all_trips.drop_duplicates()
print("dropped exact duplicates:", before - len(all_trips))

dropped exact duplicates: 0


In [ ]:
# Drop duplicate ride_ids (should be unique per trip)
before = len(all_trips)
all_trips = all_trips.drop_duplicates(subset="ride_id")
print("dropped duplicate ride_ids:", before - len(all_trips))

dropped duplicate ride_ids: 0


In [ ]:
# Drop rows missing the fields you actually need
before = len(all_trips)
all_trips = all_trips.dropna(subset=["started_at", "ended_at", "member_casual"])
print("dropped rows missing critical fields:", before - len(all_trips))

dropped rows missing critical fields: 0


In [ ]:
print(all_trips.shape)

(5552994, 13)


All three checks returned 0 dropped rows — the raw data had no
duplicates or nulls in these fields.

## Removing Out-of-Range Dates

A small number of rides have a `started_at` date in 2024, outside this analysis's 12-month scope (Jan–Dec 2025).

In [ ]:
# drop rows where started_at is in year 2024
before = len(all_trips)
all_trips = all_trips[all_trips["started_at"].dt.year != 2024]
print("dropped 2024 rows:", before - len(all_trips))

dropped 2024 rows: 53


These 53 rows are removed to keep the dataset consistent with the intended analysis period.

In [ ]:
all_trips = all_trips.copy()

## Feature Engineering

Extract `month`, `hour`, and `day_of_week` from `started_at`, and calculate
`ride_duration` in minutes as `ended_at − started_at`. These derived fields are what make it possible to compare member vs. casual ride patterns by time of day, day of week and season.

In [ ]:
# month, hour and day of week from started_at
all_trips["month"] = all_trips["started_at"].dt.month_name()
all_trips["hour"] = all_trips["started_at"].dt.hour
all_trips["day_of_week"] = all_trips["started_at"].dt.day_name()

In [ ]:
# ride_duration in minutes
all_trips["ride_duration"] = (all_trips["ended_at"] - all_trips["started_at"]).dt.total_seconds() / 60

In [ ]:
all_trips.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,month,hour,day_of_week,ride_duration
0,7569BC890583FCD7,classic_bike,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,Wacker Dr & Washington St,KA1503000072,McClurg Ct & Ohio St,TA1306000029,41.883143,-87.637242,41.892592,-87.617289,member,January,17,Tuesday,13.957950
1,013609308856B7FC,electric_bike,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,Halsted St & Wrightwood Ave,TA1309000061,Racine Ave & Belmont Ave,TA1308000019,41.929147,-87.649153,41.939743,-87.658865,member,January,15,Saturday,5.072400
2,EACACD3CE0607C0D,classic_bike,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,January,15,Thursday,11.591667
3,EAA2485BA64710D3,classic_bike,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,January,8,Thursday,3.570550
4,7F8BE2471C7F746B,electric_bike,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,January,8,Thursday,2.573817


## Handling Invalid Ride Durations

Rides with a duration of zero or negative minutes are data errors — no real ride
takes zero time. Rides longer than 24 hours are also removed: per Divvy's own
policy, bikes not returned within 24 hours are billed as lost/stolen and don't
represent genuine ride behavior.

In [ ]:
# Check how many rides have zero or negative duration
bad = all_trips[all_trips["ride_duration"] <= 0]
print("rides with duration <= 0:", len(bad))

# Check how many rides are extremely long (over 24 hours = 1440 minutes)
long_rides = all_trips[all_trips["ride_duration"] > 1440]
print("rides over 24 hours:", len(long_rides))

rides with duration <= 0: 29
rides over 24 hours: 5585


In [ ]:
before = len(all_trips)
all_trips = all_trips[(all_trips["ride_duration"] > 0) & (all_trips["ride_duration"] <= 1440)]
print("rows dropped:", before - len(all_trips))
print(all_trips.shape)

rows dropped: 5614
(5547327, 17)


## Checking the Duration Distribution

Before choosing bucket cutoffs for ride length, checking the actual distribution of `ride_duration` — median and quartiles matter more here than min/max, since they show where most of the data actually sits.

In [ ]:
print("min:", all_trips["ride_duration"].min())
print("max:", all_trips["ride_duration"].max())
print(all_trips["ride_duration"].describe())

min: 0.0007666666666666667
max: 1439.97595
count    5.547327e+06
mean     1.453417e+01
std      2.869390e+01
min      7.666667e-04
25%      5.391442e+00
50%      9.416450e+00
75%      1.652944e+01
max      1.439976e+03
Name: ride_duration, dtype: float64


## Creating Ride-Length Buckets

Group rides into four duration categories (Under 10 min, 10–30 min, 30–60 min,
Over 60 min), chosen based on the quartiles above so the buckets meaningfully
split the data rather than being arbitrary round numbers.

In [ ]:
bins = [0, 10, 30, 60, 1440]
labels = ["Under 10 min", "10-30 min", "30-60 min", "Over 60 min"]

all_trips["time_bucket"] = pd.cut(all_trips["ride_duration"], bins=bins, labels=labels)

print(all_trips["time_bucket"].value_counts())

time_bucket
Under 10 min    2935217
10-30 min       2124929
30-60 min        372984
Over 60 min      114197
Name: count, dtype: int64


## Removing Unneeded Columns

Drop `ride_id`, station names, and station IDs. `ride_id` has already served its
purpose (deduplication); station names/IDs are dropped in favor of `start_lat`/
`start_lng`/`end_lat`/`end_lng`, which are cleaner and enable geographic analysis
without the missing-data issues station fields have in the raw data.

In [ ]:
all_trips = all_trips.drop(columns=["ride_id", "start_station_name", "start_station_id", "end_station_name", "end_station_id"])
print(all_trips.columns.tolist())

['rideable_type', 'started_at', 'ended_at', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'month', 'hour', 'day_of_week', 'ride_duration', 'time_bucket']


## Exporting Cleaned Data

Reorder `member_casual` to the front for readability, then export the final
cleaned dataset to CSV for use in Tableau and BigQuery.

In [ ]:
cols = ["member_casual"] + [c for c in all_trips.columns if c != "member_casual"]
all_trips = all_trips[cols]

print(all_trips.columns.tolist())

['member_casual', 'rideable_type', 'started_at', 'ended_at', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'month', 'hour', 'day_of_week', 'ride_duration', 'time_bucket']


In [ ]:
output_path = "/content/drive/MyDrive/working data/all_trips_cleaned.csv"
all_trips.to_csv(output_path, index=False)
print("saved to", output_path)

saved to /content/drive/MyDrive/working data/all_trips_cleaned.csv
